# CropCop Track A — Global Control / Closure

Use CROPCOP_GLOBAL_PHASE=placement, then readiness, then closure. Each phase is fail-closed and consumes only artifacts from the preceding frozen gates.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

repo = Path(os.environ["CROPCOP_REPO_ROOT"]).resolve()
analysis_sha = os.environ["CROPCOP_ANALYSIS_SHA"].strip()
account_id = os.environ.get("CROPCOP_ACCOUNT_ID", "").strip()
if len(analysis_sha) != 40:
    raise RuntimeError("CROPCOP_ANALYSIS_SHA must be a full 40-character SHA")
observed = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if observed != analysis_sha:
    raise RuntimeError(f"exact analysis checkout mismatch: expected={analysis_sha}, observed={observed}")

environment_gate = repo / "journal_extension/scripts/ensure_tracka_locked_environment.py"
subprocess.run([
    sys.executable,
    str(environment_gate),
    "--repo-root", str(repo),
    "--repair",
], cwd=repo, check=True)

os.environ.setdefault("CROPCOP_NOTEBOOK_STARTED_MONOTONIC", repr(time.monotonic()))
os.environ.setdefault("CROPCOP_NOTEBOOK_HARD_LIMIT_SECONDS", "43200")
os.environ.setdefault("CROPCOP_NOTEBOOK_FINALIZATION_MARGIN_SECONDS", "3600")
print({"analysis_sha": analysis_sha, "account_id": account_id or None, "repo": str(repo)})


In [ ]:
phase = os.environ.get("CROPCOP_GLOBAL_PHASE", "").strip().lower()
if phase not in {"placement", "readiness", "closure"}:
    raise RuntimeError("CROPCOP_GLOBAL_PHASE must be placement, readiness, or closure")
global_dir = Path(os.environ["CROPCOP_GLOBAL_CONTROL_DIR"]).resolve()
global_dir.mkdir(parents=True, exist_ok=True)
placement = global_dir / "TRACKA_POSTTRAINING_PLACEMENT_FREEZE.json"
global_readiness = global_dir / "POSTTRAINING_READINESS_GATE.json"
print({"global_phase": phase, "global_control_dir": str(global_dir)})


In [ ]:
if phase == "placement":
    reports = [os.environ[f"CROPCOP_{account}_AVAILABILITY_REPORT"] for account in ("K1", "K2", "K3")]
    command = [
        sys.executable,
        str(repo / "journal_extension/scripts/freeze_tracka_v12_posttraining_placement.py"),
        "--repo-root", str(repo),
        "--analysis-source-git-commit", analysis_sha,
        "--output", str(placement),
    ]
    for report in reports:
        command.extend(["--availability-report", report])
    subprocess.run(command, cwd=repo, check=True)
    payload = json.loads(placement.read_text(encoding="utf-8"))
    print(json.dumps({"status": payload["status"], "placement_freeze_sha256": payload["placement_freeze_sha256"]}, indent=2))


In [ ]:
if phase == "readiness":
    gates = [os.environ[f"CROPCOP_{account}_READINESS"] for account in ("K1", "K2", "K3")]
    command = [
        sys.executable,
        str(repo / "journal_extension/scripts/build_tracka_v12_posttraining_readiness.py"),
        "--repo-root", str(repo),
        "--analysis-source-git-commit", analysis_sha,
        "--output", str(global_readiness),
    ]
    for gate in gates:
        command.extend(["--account-readiness", gate])
    subprocess.run(command, cwd=repo, check=True)
    payload = json.loads(global_readiness.read_text(encoding="utf-8"))
    print(json.dumps({
        "status": payload["status"],
        "state_count": payload["state_count"],
        "unique_selected_checkpoint_count": payload["unique_selected_checkpoint_count"],
        "gate_sha256": payload["gate_sha256"],
    }, indent=2))


In [ ]:
if phase == "closure":
    account_completions = [os.environ[f"CROPCOP_{account}_COMPLETION"] for account in ("K1", "K2", "K3")]
    audit_dir = global_dir / "evidence_audit"
    if audit_dir.exists():
        raise RuntimeError("evidence_audit directory already exists; closure phase is append-only")
    command = [
        sys.executable,
        str(repo / "journal_extension/scripts/audit_tracka_v12_posttraining_evidence.py"),
        "--repo-root", str(repo),
        "--analysis-source-git-commit", analysis_sha,
        "--global-readiness", str(global_readiness),
        "--output-dir", str(audit_dir),
    ]
    for manifest in account_completions:
        command.extend(["--account-completion", manifest])
    subprocess.run(command, cwd=repo, check=True)
    closure_dir = global_dir / "final_closure"
    subprocess.run([
        sys.executable,
        str(repo / "journal_extension/scripts/close_tracka_v12.py"),
        "--repo-root", str(repo),
        "--analysis-source-git-commit", analysis_sha,
        "--global-evidence-audit", str(audit_dir / "TRACKA_POSTTRAINING_GLOBAL_EVIDENCE_AUDIT.json"),
        "--direct-evidence-index", str(audit_dir / "TRACKA_DIRECT_EVIDENCE_INDEX.json"),
        "--auxiliary-evidence-index", str(audit_dir / "TRACKA_AUXILIARY_EVIDENCE_INDEX.json"),
        "--output-dir", str(closure_dir),
    ], cwd=repo, check=True)
    final = json.loads((closure_dir / "TRACKA_FINAL_CLOSURE_AUDIT.json").read_text(encoding="utf-8"))
    if final.get("status") != "PASS" or final.get("track_a_closed") is not True:
        raise RuntimeError("final Track-A closure audit is not PASS")
    print(json.dumps(final, indent=2, sort_keys=True))
